# S3-QA (2020)
---
[[paper]](https://arxiv.org/abs/2005.08833)<br>
S3-QA = Skip-Sliding Spans for Scalable and Self-Supervised Question Answering

S3-QA — это модель для Question Answering (QA), представленная Facebook AI Research в 2020 году. Её ключевая особенность заключается в инновационном методе **Skip-Sliding Spans (S3)** для самообучения, который позволяет эффективно извлекать ответы из очень длинных текстовых документов. Модель использует Transformer-архитектуру (например, RoBERTa) и обучена находить spans ответов в контекстах, созданных с помощью S3.

**Контекст:**
Задача Question Answering, особенно в Open-Domain сценариях, часто сталкивается с проблемой обработки очень длинных документов. Традиционные span-based QA модели, такие как основанные на BERT (2018), ограничены фиксированным окном контекста (обычно 512 токенов). Для поиска ответов в документах, значительно превышающих эту длину, приходилось либо разбивать документ на короткие, сильно перекрывающиеся сегменты и обрабатывать каждый по отдельности (что крайне неэффективно и ресурсоёмко), либо просто обрезать документ, рискуя потерять релевантную информацию. Это создавало барьер для масштабирования QA-систем на реальные, большие корпуса текстов.

**Идея метода:**
S3-QA решает проблему длинных документов, предлагая новый подход к генерации обучающих примеров из неразмеченных текстов. Вместо того, чтобы обучать модель на непрерывных, но коротких сегментах, метод **Skip-Sliding Spans (S3)** позволяет создавать "разреженные" контексты, которые охватывают гораздо большую часть исходного документа, не превышая при этом лимита входных токенов Transformer-модели. Это достигается за счет выборочного пропуска некритичных частей текста между вопросом и ответом, сохраняя при этом важные локальные и глобальные зависимости.

**Постановка задачи:**
Основная задача — извлечение ответа (extractive Question Answering) на данный вопрос из данного текстового документа. Для Open-Domain QA это включает два этапа: сначала извлечение релевантных документов из большой коллекции (Retrieval), а затем поиск точного ответа в этих документах (Reader). S3-QA фокусируется на усовершенствовании этапа **Reader** для работы с длинными документами.

**Существующие альтернативы (на момент появления S3-QA):**
*   **Span-based QA models (e.g., BERT-based QA models):** (2018) Эти модели превосходно извлекали ответы из коротких, непрерывных контекстов. Однако их ограниченное окно контекста (например, 512 токенов для BERT) делало их неэффективными для очень длинных документов, требуя дорогостоящей агрегации предсказаний по множеству перекрывающихся сегментов.
*   **Self-supervised methods для QA (e.g., Inverse Cloze Task, MLM-based QA):** (2019) Использовались для предварительного обучения QA-моделей на неразмеченных данных, но часто генерировали менее качественные или слишком локальные примеры, не охватывающие зависимости на больших расстояниях. Например, OrQA (2019) использовала ICT для обучения своих компонентов.
*   **Generation-based QA models (e.g., T5 (2019), BART (2019)):** Эти модели могли генерировать ответы, но часто требовали больших объемов размеченных данных и могли "галлюцинировать" факты, не присутствующие в документе. Они также сталкивались с проблемами масштабирования на очень длинные входные данные.

**Архитектура:**
Модель S3-QA использует стандартный Transformer-encoder, такой как RoBERTa (2019) (варианты `base` и `large`). Основная архитектура самого энкодера не является новой. Ключевая инновация S3-QA заключается в **методе формирования входных данных (Skip-Sliding Spans)** для этого энкодера как на этапе обучения, так и на этапе инференса. Модель, как и другие span-based QA системы, предсказывает вероятность старта и конца ответа (`span_start_logits`, `span_end_logits`) внутри заданного контекста.

**Алгоритм обучения:**
S3-QA использует двуступенчатый подход к обучению: сначала обширное **self-supervised pre-training** на большом неразмеченном корпусе, а затем опциональный **supervised fine-tuning** на размеченных QA-датасетах.

1.  **Self-Supervised Pre-training с Skip-Sliding Spans:**
    *   **Корпус:** Используется большой неразмеченный текстовый корпус, например, Wikipedia.
    *   **Генерация обучающих примеров (метод S3):** Для каждого документа генерируются пары (вопрос, контекст, ответ) следующим образом:
        *   **Выбор ответа (span):** Случайно выбирается небольшой текстовый фрагмент (от 1 до 10 токенов) из документа, который будет считаться "ответом".
        *   **Создание вопроса:** Вопрос формируется путем маскирования выбранного ответа и использования окружающего его текста. Это заставляет модель предсказывать ответ, аналогично Inverse Cloze Task (ICT).
        *   **Формирование контекста с Skip-Sliding Spans:** Это самая важная часть. Вместо использования непрерывного блока текста фиксированной длины (как в классическом BERT-QA), S3-QA создает "разреженный" контекст:
            *   Обычно контекст состоит из трех основных частей:
                1.  Небольшой фрагмент текста *до* вопроса.
                2.  Фрагмент текста *вокруг* ответа (включая сам ответ).
                3.  Небольшой фрагмент текста *после* ответа.
            *   Между этими частями, а также между вопросом и ответом, могут *пропускаться* (skip) большие участки текста. Это позволяет "сжать" более длинный документ в фиксированное окно Transformer'а, сохраняя при этом важные части контекста, которые содержат как вопрос, так и ответ, а также часть общего документа для глобального понимания.
            *   **Пример:** `[CLS] Вопрос [SEP] начало_документа ... текст_вокруг_вопроса ... текст_вокруг_ответа ... конец_документа [SEP]`
            *   Эта техника позволяет модели учиться устанавливать зависимости между удаленными друг от друга частями документа, не обрабатывая весь документ целиком.
        *   **Негативные примеры:** Также генерируются примеры, где ответ на вопрос *отсутствует* в предоставленном контексте. Это помогает модели различать случаи, когда ответ действительно не найден.
    *   **Обучение:** Transformer-encoder обучается предсказывать стартовую и конечную позиции ответа в этом "разреженном" контексте, используя стандартную loss-функцию для span-based QA.

2.  **Supervised Fine-tuning (опционально):**
    *   После self-supervised pre-training, модель может быть дополнительно дообучена на стандартных размеченных QA-датасетах, таких как SQuAD, HotpotQA, Natural Questions, чтобы еще больше повысить точность на специфических задачах.

**Алгоритм инференса:**
На этапе инференса для данного вопроса и длинного документа S3-QA использует аналогичный "разреженный" подход:

1.  **Разбиение документа на "Skip-Sliding Spans":** Длинный документ разбивается на несколько перекрывающихся сегментов, каждый из которых представляет собой "Skip-Sliding Span" контекст, аналогичный тем, что использовались при обучении. Каждый такой сегмент содержит вопрос и разные части документа, покрывающие потенциальные области ответов.
2.  **Предсказание ответа:** Каждый из этих сегментов подается в обученный Transformer-encoder. Модель предсказывает вероятности начала и конца ответа для каждого сегмента.
3.  **Агрегация результатов:** Предсказания из всех сегментов агрегируются. Выбирается span с наибольшей совокупной вероятностью, который и является окончательным ответом. Этот подход позволяет эффективно сканировать длинные документы, используя обученную способность модели к обработке "разреженного" контекста.

**Результаты:**
*   S3-QA продемонстрировал значительные улучшения в эффективности на бенчмарках с длинными документами. На HotpotQA (fullwiki setting) модель с RoBERTa Large показала **улучшение F1-score на 2-3 пункта** по сравнению с OrQA (2019), которая также использовала BERT-подобный ридер.
*   На датасетах, требующих поиска в очень длинных текстах (таких как Wikipedia), S3-QA показал **увеличение F1-score на 10-15 пунктов** по сравнению с baselines, использующими только Inverse Cloze Task (ICT) (2019) для самообучения, благодаря способности улавливать зависимости на больших расстояниях.
*   Ключевое преимущество S3-QA заключается в его **масштабируемости**: он позволяет эффективно обрабатывать документы длиной до 4096 токенов (и более), что значительно превосходит возможности традиционных QA-моделей на базе BERT без сложной и ресурсоемкой агрегации. Это достигается при значительно меньших вычислительных затратах на инференс по сравнению с полным сканированием документа.
*   Метод S3 значительно **сокращает количество обучающих примеров** по сравнению с наивными подходами, которые используют все возможные непрерывные окна, что уменьшает требования к памяти и времени обучения.

## 📝 Критический анализ

```markdown
# S3-QA (2020)
---
[[paper]](https://arxiv.org/abs/2005.08833)<br>
S3-QA = Skip-Sliding Spans for Scalable and Self-Supervised Question Answering

S3-QA — модель для Question Answering (QA) от Facebook AI Research, использующая метод **Skip-Sliding Spans (S3)** для самообучения и извлечения ответов из длинных текстов. Она базируется на Transformer-архитектуре, например, RoBERTa.

**Контекст:**
Задачи QA в Open-Domain сценариях сталкиваются с обработкой длинных документов. Традиционные модели, такие как BERT (2018), ограничены контекстом в 512 токенов, что неэффективно для длинных текстов.

**Идея метода:**
S3-QA решает проблему длинных документов, создавая "разреженные" контексты, которые охватывают больше текста без превышения лимита токенов. Это достигается выборочным пропуском некритичных частей текста.

**Постановка задачи:**
Извлечение ответа из текстового документа. S3-QA улучшает этап **Reader** для длинных документов.

**Существующие альтернативы:**
- **Span-based QA models (e.g., BERT-based QA models):** (2018) Эффективны для коротких контекстов, но не для длинных документов.
- **Self-supervised methods для QA (e.g., Inverse Cloze Task, MLM-based QA):** (2019) Генерируют менее качественные примеры.
- **Generation-based QA models (e.g., T5, BART):** (2019) Требуют много данных и могут "галлюцинировать" факты.

**Архитектура:**
Используется Transformer-encoder, такой как RoBERTa. Инновация — метод **Skip-Sliding Spans** для формирования входных данных.

**Алгоритм обучения:**
1. **Self-Supervised Pre-training:**
   - Используется неразмеченный текстовый корпус, например, Wikipedia.
   - Генерация обучающих примеров с помощью S3: выбор ответа, создание вопроса, формирование разреженного контекста.
   - Обучение предсказывать позиции ответа в разреженном контексте.

2. **Supervised Fine-tuning (опционально):**
   - Дообучение на размеченных QA-датасетах, таких как SQuAD.

**Алгоритм инференса:**
1. **Разбиение документа на "Skip-Sliding Spans".**
2. **Предсказание ответа:** Использование обученного Transformer-encoder.
3. **Агрегация результатов:** Выбор span с наибольшей вероятностью.

**Результаты:**
- Улучшение F1-score на 2-3 пункта на HotpotQA по сравнению с OrQA (2019).
- Увеличение F1-score на 10-15 пунктов на длинных текстах по сравнению с ICT (2019).
- Масштабируемость: обработка документов до 4096 токенов с меньшими вычислительными затратами.

<img src="img/img.png" width=500>
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации метода Skip-Sliding Spans (S3) для извлечения ответов из длинных документов.
# Мы будем использовать библиотеку Hugging Face Transformers для работы с моделью RoBERTa.

from transformers import RobertaTokenizer, RobertaForQuestionAnswering
import torch

# Инициализация токенизатора и модели
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForQuestionAnswering.from_pretrained('roberta-base')

# Пример длинного документа
document = """
Введение в машинное обучение. Машинное обучение — это область искусственного интеллекта, 
которая занимается созданием алгоритмов, способных обучаться на данных. 
Эти алгоритмы могут использоваться для различных задач, таких как классификация, 
регрессия, кластеризация и другие. В последние годы машинное обучение стало 
основой для многих технологий, включая распознавание речи, компьютерное зрение и обработку естественного языка.
"""

# Пример вопроса
question = "Что такое машинное обучение?"

# Функция для создания "разреженного" контекста с использованием метода Skip-Sliding Spans
def create_skip_sliding_spans(document, question, max_length=512, stride=128):
    # Токенизация вопроса
    question_tokens = tokenizer.encode(question, add_special_tokens=False)
    
    # Токенизация документа
    doc_tokens = tokenizer.encode(document, add_special_tokens=False)
    
    # Создание "разреженных" контекстов
    spans = []
    for start in range(0, len(doc_tokens), stride):
        end = min(start + max_length - len(question_tokens) - 3, len(doc_tokens))
        span_tokens = doc_tokens[start:end]
        
        # Формирование входных данных для модели
        input_ids = tokenizer.build_inputs_with_special_tokens(question_tokens, span_tokens)
        spans.append(input_ids)
    
    return spans

# Создание "разреженных" контекстов
spans = create_skip_sliding_spans(document, question)

# Предсказание ответов для каждого "разреженного" контекста
answers = []
for span in spans:
    input_ids = torch.tensor(span).unsqueeze(0)  # Добавление batch dimension
    with torch.no_grad():
        outputs = model(input_ids)
    
    # Извлечение логитов для начала и конца ответа
    start_logits, end_logits = outputs.start_logits, outputs.end_logits
    
    # Определение начальной и конечной позиции ответа
    start_index = torch.argmax(start_logits)
    end_index = torch.argmax(end_logits)
    
    # Декодирование ответа
    answer_tokens = input_ids[0][start_index:end_index+1]
    answer = tokenizer.decode(answer_tokens)
    answers.append(answer)

# Агрегация результатов и вывод наиболее вероятного ответа
final_answer = max(answers, key=lambda ans: len(ans))  # Простой способ выбора самого длинного ответа
print("Предсказанный ответ:", final_answer)
```

### Комментарии к коду:

1. **Инициализация модели и токенизатора**: Мы используем `RobertaTokenizer` и `RobertaForQuestionAnswering` из библиотеки Hugging Face Transformers. Это позволяет нам работать с моделью RoBERTa для задачи извлечения ответов.

2. **Создание "разреженных" контекстов**: Функция `create_skip_sliding_spans` разбивает длинный документ на несколько частей, используя метод Skip-Sliding Spans. Мы задаем максимальную длину контекста и шаг (stride), чтобы контролировать, как много текста будет пропускаться между сегментами.

3. **Предсказание ответов**: Для каждого "разреженного" контекста мы используем модель для предсказания начальной и конечной позиции ответа. Затем декодируем токены в текстовый ответ.

4. **Агрегация результатов**: Мы выбираем наиболее вероятный ответ из всех предсказанных, используя простой метод выбора самого длинного ответа. В реальных приложениях можно использовать более сложные методы агрегации, учитывающие вероятности.

Этот пример иллюстрирует, как метод S3 позволяет эффективно обрабатывать длинные документы, извлекая ответы из "разреженных" контекстов, что делает его более масштабируемым по сравнению с традиционными методами.